# CEO Letter / Annual Report Event Study — US Insurers (First Pass)

Tests whether abnormal stock returns around CEO letter/annual report publication dates correlate
with ESG narrative specificity. **US-region firms only for this first pass** (7 companies x 12
fiscal years = 84 firm-years). Kept **fully separate** from
`PHDp2_CEOLetters_AnnualReports_TextAnalysis.ipynb` and `PHDp2_FinancialData_Regression.ipynb` --
this notebook does not import from or modify either. Not validated yet -- developed on branch
`event-study`, not `main`.

**Report-only pipeline.** No merge with specificity measures, no regression -- Step 5 ends with a
confirmed firm-year event-study panel, full stop.

## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

# pip installs yfinance if not already present in this Colab runtime
try:
    import yfinance as yf
except ImportError:
    %pip install -q yfinance
    import yfinance as yf

print("yfinance version:", yf.__version__)


## Step 1 — Load event dates

Read `Combined_Insurer_Publication_Dates.xlsx`, filter to `Region == "United States"`, confirm 84
firm-years (7 companies x 12 fiscal years) before proceeding. Chubb's rows already unify the
2016 ACE Limited -> Chubb Limited name change into a single `"Chubb"` company label in the source
file (confirmed directly: FY2012-2015 rows carry a note "Filed as ACE Limited pre-2016 merger,"
FY2016-2023 rows carry "Filed as Chubb Limited" -- both already grouped under `Company == "Chubb"`
with `Region == "United States"`), so no company-name unification step is needed here -- it's
already done in the source data. The ticker-level ACE/CB split is handled in Step 2/3 instead,
since that's a market-data concern, not an event-date concern.

In [ ]:
# Adjust this path if the file lives elsewhere in your Drive.
event_dates_path = "/content/drive/MyDrive/phd/Data/Combined_Insurer_Publication_Dates.xlsx"

df_dates_raw = pd.read_excel(event_dates_path)
print(f"Full file: {df_dates_raw.shape}")
print("Regions present:", sorted(df_dates_raw['Region'].unique()))

df_us = df_dates_raw[df_dates_raw["Region"] == "United States"].copy()
print(f"\nUS-region rows: {len(df_us)}")

# ------------------------------------------------------------
# Confirm 84 firm-years (7 companies x 12 fiscal years) before proceeding,
# per instruction -- stop and investigate if this doesn't hold, rather
# than silently continue on a wrong filter.
# ------------------------------------------------------------
n_companies = df_us["Company"].nunique()
n_years = df_us["Fiscal Year"].nunique()
print(f"Distinct companies: {n_companies}")
print(f"Distinct fiscal years: {n_years}")
print(f"Companies: {sorted(df_us['Company'].unique())}")
print(f"Fiscal years: {sorted(df_us['Fiscal Year'].unique())}")

counts_per_company = df_us["Company"].value_counts()
print("\nRows per company:")
print(counts_per_company)

assert len(df_us) == 84, f"Expected 84 US firm-years, got {len(df_us)} -- STOP, investigate before continuing."
assert n_companies == 7, f"Expected 7 US companies, got {n_companies}"
assert (counts_per_company == 12).all(), "Not every company has exactly 12 fiscal years -- STOP."
print("\nCONFIRMED: 84 US firm-years (7 companies x 12 fiscal years).")

# ------------------------------------------------------------
# Parse Publication Date (stored as a string in the source file, not a
# native Excel date) and sanity-check the Chubb/ACE transition boundary
# noted above.
# ------------------------------------------------------------
df_us["Publication Date"] = pd.to_datetime(df_us["Publication Date"])
print(f"\nPublication date range: {df_us['Publication Date'].min()} to {df_us['Publication Date'].max()}")

chubb = df_us[df_us["Company"] == "Chubb"][["Fiscal Year", "Publication Date", "Note"]].sort_values("Fiscal Year")
print("\nChubb rows (confirm ACE->Chubb transition sits between FY2015 and FY2016):")
print(chubb.to_string(index=False))


## Step 2 — Tickers

Company -> ticker mapping, plus the benchmark (`URTH`, iShares MSCI World ETF). Chubb needs
special handling: pre-2016 it traded as **ACE Limited** under ticker **ACE**, not `CB`. Rather
than assume where `CB`'s history starts, this pulls both `ACE` and `CB` and empirically checks
where each series actually starts/ends -- the actual data boundary, not an assumed merger-closing
date, decides the splice point (handled in Step 3).

In [ ]:
COMPANY_TICKER = {
    "American International Group (AIG)": "AIG",
    "Chubb": "CB",                     # current ticker; ACE spliced in for pre-merger history
    "MetLife, Inc.": "MET",
    "Prudential Financial, Inc.": "PRU",
    "The Allstate Corporation": "ALL",
    "The Progressive Corporation": "PGR",
    "The Travelers Companies, Inc.": "TRV",
}
ACE_TICKER = "ACE"          # Chubb's pre-2016-merger ticker (as ACE Limited)
BENCHMARK_TICKER = "URTH"   # iShares MSCI World ETF

# ACE is checked SEPARATELY and non-fatally below -- it's Chubb's
# pre-2016 ticker, and Yahoo Finance frequently drops a ticker's price
# history entirely after a symbol change/delisting rather than
# archiving it, so ACE failing to resolve at all (confirmed: fails even
# against a 2014 window, when it was definitely actively trading) is a
# real possible outcome, not necessarily a bug to "fix." The other 8
# tickers (7 companies' current symbols + URTH) are the ones that must
# all resolve for this pipeline to proceed at all.
required_tickers = list(COMPANY_TICKER.values()) + [BENCHMARK_TICKER]
print("Required tickers to verify:", required_tickers)

DEFAULT_PROBE_WINDOW = ("2019-01-01", "2019-01-31")

ticker_resolves = {}
for ticker in required_tickers:
    start, end = DEFAULT_PROBE_WINDOW
    probe = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
    ok = len(probe) > 0
    ticker_resolves[ticker] = ok
    print(f"{ticker}: {'OK' if ok else 'FAILED'} ({len(probe)} rows in {start} probe window)")

failed = [t for t, ok in ticker_resolves.items() if not ok]
if failed:
    raise RuntimeError(f"These REQUIRED tickers did not resolve in yfinance: {failed} -- STOP, fix before Step 3.")
print("\nAll required tickers resolve.")

# ------------------------------------------------------------
# ACE probe (2014, when it was definitely trading) -- reported, not
# raised on, since the real question this pipeline needs answered is
# "does SOME source cover the pre-2016 Chubb event dates," and CB's own
# history (checked next) is the other candidate.
# ------------------------------------------------------------
ace_probe = yf.download("ACE", start="2014-01-01", end="2014-01-31", progress=False, auto_adjust=True)
ace_probe_ok = len(ace_probe) > 0
print(f"\nACE (informational, not required): {'OK' if ace_probe_ok else 'NO DATA'} "
      f"({len(ace_probe)} rows in 2014-01 probe window)")

# ------------------------------------------------------------
# Check whether CB's own history already extends back far enough to
# cover the earliest Chubb event dates (2012-2015 fiscal years, i.e.
# publication dates as early as 2013). Report the actual first
# available date rather than assume -- this determines whether the
# pre-2016 Chubb firm-years are usable at all via yfinance.
# ------------------------------------------------------------
cb_full = yf.download("CB", start="2011-01-01", end="2016-06-30", progress=False, auto_adjust=True)
ace_full = yf.download("ACE", start="2011-01-01", end="2016-06-30", progress=False, auto_adjust=True) if ace_probe_ok else None

print(f"\nCB: first available date = {cb_full.index.min() if len(cb_full) else 'NO DATA'}")
if ace_full is not None and len(ace_full):
    print(f"ACE: first available date = {ace_full.index.min()}, last available date = {ace_full.index.max()}")
else:
    print("ACE: NO DATA available from yfinance at all (confirmed via both the 2014 and "
          "2011-2016 pulls) -- Yahoo Finance has evidently dropped this ticker's history "
          "entirely rather than archiving it under the old symbol.")

earliest_chubb_event = chubb["Publication Date"].min()
print(f"\nEarliest Chubb event date requiring price history: {earliest_chubb_event}")
print(f"Estimation window for that event needs data back to roughly "
      f"{earliest_chubb_event - pd.Timedelta(days=380)} (250 trading days plus weekends/holidays buffer)")

CB_COVERS_PRE_2016 = len(cb_full) > 0 and cb_full.index.min() <= pd.Timestamp("2015-01-01")
ACE_AVAILABLE = ace_full is not None and len(ace_full) > 0

if CB_COVERS_PRE_2016:
    print("\nCB's own history extends back far enough -- no splice needed. VERIFY this isn't "
          "Yahoo silently truncating/misdating data rather than genuinely having ACE-era prices "
          "under the CB symbol, by spot-checking one known pre-2016 price point before trusting it.")
    CHUBB_STRATEGY = "cb_only"
elif ACE_AVAILABLE:
    print("\nCB does NOT cover pre-2016, but ACE does -- splicing ACE + CB (handled in Step 4).")
    CHUBB_STRATEGY = "splice"
else:
    print("\n" + "=" * 70)
    print("DATA GAP: neither CB nor ACE covers Chubb's pre-2016 fiscal years via yfinance.")
    print("=" * 70)
    print("This means Chubb FY2012-2015 (4 of the 84 firm-years) cannot be included in the "
          "event study through this data source. Two options, NOT decided here:")
    print("  (a) Drop those 4 Chubb firm-years, proceed with N=80 (84 - 4).")
    print("  (b) Source ACE's pre-2016 price history from a different provider (e.g. a paid "
          "data vendor, or a manually-compiled CSV) and splice it in manually.")
    print("Proceeding with CB-only for Chubb (FY2016-2023) and flagging FY2012-2015 as excluded "
          "in Step 4/5's output -- report back and we can revisit if (b) is wanted instead.")
    CHUBB_STRATEGY = "cb_only_with_gap"

print(f"\nCHUBB_STRATEGY = {CHUBB_STRATEGY!r}")


## Step 3 — Pull daily price data

All 7 tickers (+ `ACE` for the pre-2016 Chubb splice) + `URTH`, Jan 2011 (estimation-window
buffer before the earliest 2012 fiscal-year event) through Dec 2023. Raw pulls saved to disk
before any processing, so Step 4/5 can be re-run without re-hitting the API.

In [ ]:
PRICE_START = "2011-01-01"
# 2023-12-31 originally -- WRONG: FY2023 events are published in Q1/Q2
# 2024 (latest event date 2024-04-26), so that cutoff excluded every
# FY2023 event window and part of some FY2023 estimation windows too.
# Confirmed systematic across all 7 companies (not Chubb-specific) via
# the exclusion-accounting re-check in the report-only summary cell.
# Extended to give buffer past the latest event date's post-event window.
PRICE_END = "2024-07-31"

raw_prices = {}
tickers_to_pull = list(COMPANY_TICKER.values()) + [BENCHMARK_TICKER]
if CHUBB_STRATEGY == "splice":
    tickers_to_pull.append(ACE_TICKER)
else:
    print(f"CHUBB_STRATEGY = {CHUBB_STRATEGY!r} -- not pulling ACE (unavailable or unneeded).")

for ticker in tickers_to_pull:
    if ticker in raw_prices:
        continue  # CB only pulled once even though it's used for one company
    print(f"Pulling {ticker}...")
    data = yf.download(ticker, start=PRICE_START, end=PRICE_END, progress=False, auto_adjust=True)
    raw_prices[ticker] = data
    print(f"  {len(data)} rows, {data.index.min()} to {data.index.max()}")

raw_prices_dir = "/content/drive/MyDrive/phd/Data/event_study_raw_prices"
os.makedirs(raw_prices_dir, exist_ok=True)
for ticker, data in raw_prices.items():
    out_path = os.path.join(raw_prices_dir, f"{ticker}.csv")
    data.to_csv(out_path)
    print(f"Saved {ticker} -> {out_path}")

print(f"\nAll raw pulls saved to: {raw_prices_dir}")


## Step 4 — Market-model event study, per firm-year

Estimation window `[-250, -30]` trading days relative to the (trading-day-adjusted) event date;
OLS of firm return on `URTH` return -> firm-specific alpha, beta. Event windows: primary
`[-1, +1]`; also `[-2, +2]` and `[0, +1]` as robustness alternatives, all three reported.
Abnormal return = actual - (alpha + beta x URTH return); CAR = sum over the window, both signed
and `|CAR|` reported. Non-trading-day publication dates shift to the next available trading day
(reported how many). Firm-years with fewer than 100 valid estimation-window trading days excluded
(reported which, if any).

In [ ]:
# ------------------------------------------------------------
# Build one continuous return series per company. For Chubb, splice ACE
# (up to ACE's last available date) and CB (from CB's first available
# date) -- using the ACTUAL empirical data boundary from Step 2, not an
# assumed merger-closing date, so the estimation window for the FY2015
# event (published 2016-04-08, whose [-250,-30] estimation window
# extends back into 2015, i.e. BEFORE the ACE->CB ticker switch) is
# built from genuinely continuous, correctly-sourced price data rather
# than silently using CB data that doesn't exist yet for that period.
# ------------------------------------------------------------
def get_adj_close(ticker):
    # yfinance can return either flat columns or MultiIndex columns
    # (ticker as a sub-level) even for a single-ticker download,
    # depending on version -- handled robustly rather than assumed flat,
    # since a silent DataFrame-vs-Series mismatch here would break
    # everything downstream without an obvious error at the point of failure.
    close = raw_prices[ticker]["Close"]  # auto_adjust=True -> Close is already adjusted
    if isinstance(close, pd.DataFrame):
        close = close.squeeze("columns")
    return close.dropna()

def build_return_series(company):
    ticker = COMPANY_TICKER[company]
    if company == "Chubb" and CHUBB_STRATEGY == "splice":
        ace_close = get_adj_close(ACE_TICKER)
        cb_close = get_adj_close("CB")
        splice_date = cb_close.index.min()
        ace_part = ace_close[ace_close.index < splice_date]
        print(f"Chubb: splicing ACE (through {ace_part.index.max()}) "
              f"+ CB (from {cb_close.index.min()})")
        gap_days = (cb_close.index.min() - ace_part.index.max()).days
        if gap_days > 10:
            print(f"  WARNING: {gap_days}-day gap between ACE's last price and CB's first price "
                  f"-- investigate before trusting the spliced return series across this boundary.")
        combined_close = pd.concat([ace_part, cb_close]).sort_index()
        combined_close = combined_close[~combined_close.index.duplicated(keep="last")]
    elif company == "Chubb":
        # CHUBB_STRATEGY is "cb_only" or "cb_only_with_gap" -- CB history
        # only, no ACE splice. In the "_with_gap" case this means
        # FY2012-2015 events will have no estimation-window data
        # available and get excluded naturally by Step 4's existing
        # "estimation window falls outside available price history"
        # check below -- not special-cased here, just a consequence of
        # only having CB's actual price history to work with.
        combined_close = get_adj_close("CB")
        print(f"Chubb: CB only (CHUBB_STRATEGY={CHUBB_STRATEGY!r}), "
              f"history from {combined_close.index.min()}")
    else:
        combined_close = get_adj_close(ticker)
    returns = combined_close.pct_change().dropna()
    returns.name = company
    return returns

benchmark_returns = get_adj_close(BENCHMARK_TICKER).pct_change().dropna()
benchmark_returns.name = "URTH"

company_returns = {company: build_return_series(company) for company in COMPANY_TICKER}


In [ ]:
# ------------------------------------------------------------
# Trading-day calendar taken directly from URTH's own observed trading
# days (rather than a generic calendar), so "next available trading day"
# and window offsets are defined consistently with the actual price data
# being used.
# ------------------------------------------------------------
trading_days = benchmark_returns.index.sort_values()

def next_trading_day(date, calendar):
    idx = calendar.searchsorted(date)
    if idx >= len(calendar):
        return None
    return calendar[idx]

def trading_day_offset(date, calendar, offset):
    """Trading day `offset` sessions away from `date` (date must be IN calendar)."""
    pos = calendar.get_loc(date)
    new_pos = pos + offset
    if new_pos < 0 or new_pos >= len(calendar):
        return None
    return calendar[new_pos]

EVENT_WINDOWS = {
    "primary_-1_+1": (-1, 1),
    "robustness_-2_+2": (-2, 2),
    "robustness_0_+1": (0, 1),
    "robustness_0_+5": (0, 5),
    "robustness_0_+20": (0, 20),
}
ESTIMATION_WINDOW = (-250, -30)
MIN_ESTIMATION_OBS = 100

results = []
n_shifted = 0
excluded_thin_estimation = []

for _, row in df_us.iterrows():
    company = row["Company"]
    fiscal_year = row["Fiscal Year"]
    raw_event_date = row["Publication Date"]

    firm_ret = company_returns[company]
    # returns are pct_change of price, indexed by the LATER of the two
    # price dates -- align against a calendar of dates where a return is
    # actually observable for this firm.
    firm_calendar = firm_ret.index

    # Shift to next available trading day if the raw event date isn't one.
    if raw_event_date in firm_calendar:
        event_date = raw_event_date
    else:
        event_date = next_trading_day(raw_event_date, firm_calendar)
        n_shifted += 1
        if event_date is None:
            print(f"SKIPPING {company} FY{fiscal_year}: no trading day found on/after "
                  f"{raw_event_date} in the available price history.")
            continue

    event_pos_check = firm_calendar.get_loc(event_date)

    # ---- estimation window ----
    est_start = trading_day_offset(event_date, firm_calendar, ESTIMATION_WINDOW[0])
    est_end = trading_day_offset(event_date, firm_calendar, ESTIMATION_WINDOW[1])
    if est_start is None or est_end is None:
        print(f"SKIPPING {company} FY{fiscal_year}: estimation window falls outside available "
              f"price history.")
        continue

    est_mask = (firm_ret.index >= est_start) & (firm_ret.index <= est_end)
    est_firm = firm_ret[est_mask]
    est_bench = benchmark_returns.reindex(est_firm.index).dropna()
    est_common = est_firm.index.intersection(est_bench.index)
    est_firm = est_firm.loc[est_common]
    est_bench = benchmark_returns.loc[est_common]

    n_est = len(est_common)
    if n_est < MIN_ESTIMATION_OBS:
        excluded_thin_estimation.append((company, fiscal_year, n_est))
        continue

    # OLS: firm_return = alpha + beta * bench_return
    X = np.column_stack([np.ones(n_est), est_bench.values])
    coefs, _, _, _ = np.linalg.lstsq(X, est_firm.values, rcond=None)
    alpha, beta = coefs[0], coefs[1]

    # ---- event windows ----
    car_values = {}
    for win_name, (lo, hi) in EVENT_WINDOWS.items():
        win_start = trading_day_offset(event_date, firm_calendar, lo)
        win_end = trading_day_offset(event_date, firm_calendar, hi)
        if win_start is None or win_end is None:
            car_values[win_name] = np.nan
            continue
        win_mask = (firm_ret.index >= win_start) & (firm_ret.index <= win_end)
        win_firm = firm_ret[win_mask]
        win_bench = benchmark_returns.reindex(win_firm.index)
        abnormal = win_firm - (alpha + beta * win_bench)
        car_values[win_name] = abnormal.sum()

    results.append({
        "Company": company,
        "Fiscal Year": fiscal_year,
        "Event Date (raw)": raw_event_date,
        "Event Date (adjusted)": event_date,
        "Shifted": event_date != raw_event_date,
        "Estimation N": n_est,
        "Alpha": alpha,
        "Beta": beta,
        "CAR[-1,+1]": car_values["primary_-1_+1"],
        "CAR[-2,+2]": car_values["robustness_-2_+2"],
        "CAR[0,+1]": car_values["robustness_0_+1"],
        "CAR[0,+5]": car_values["robustness_0_+5"],
        "CAR[0,+20]": car_values["robustness_0_+20"],
        "|CAR|[-1,+1]": abs(car_values["primary_-1_+1"]),
    })

print(f"Publication dates shifted to next trading day: {n_shifted} of {len(df_us)}")
print(f"\nFirm-years excluded for <{MIN_ESTIMATION_OBS} estimation-window observations: "
      f"{len(excluded_thin_estimation)}")
for company, fy, n in excluded_thin_estimation:
    print(f"  {company} FY{fy}: only {n} estimation-window observations")


## Step 5 — Firm-year event-study panel

`Company | Fiscal Year | Event Date (adjusted) | Estimation N | Alpha | Beta | CAR[-1,+1] |
CAR[-2,+2] | CAR[0,+1] | |CAR|[-1,+1]`. No merge with specificity measures, no regression -- this
panel, confirmed correct, is the end of this notebook's first pass.

In [ ]:
panel = pd.DataFrame(results)
panel = panel.sort_values(["Company", "Fiscal Year"]).reset_index(drop=True)

print(f"Firm-years in final panel: {len(panel)} of {len(df_us)} original US firm-years")
print(f"({len(df_us) - len(panel)} excluded -- see Step 4's exclusion list above for why)")

display_cols = ["Company", "Fiscal Year", "Event Date (adjusted)", "Estimation N",
                "Alpha", "Beta", "CAR[-1,+1]", "CAR[-2,+2]", "CAR[0,+1]",
                "CAR[0,+5]", "CAR[0,+20]", "|CAR|[-1,+1]"]
display(panel[display_cols])

output_path = "/content/drive/MyDrive/phd/Data/ceo_letter_event_study_panel_US.csv"
panel[display_cols].to_csv(output_path, index=False)
print(f"\nSaved panel to: {output_path}")


## Report-only summary

CAR summary stats, exclusions and why, confirmation of the Chubb/ACE handling, and a spot-check
for unadjusted stock splits (an obviously-wrong single-day return, e.g. close to -50% or +100%,
is the classic signature of a split that `auto_adjust=True` failed to correctly adjust for --
checked explicitly below rather than assumed fine).

In [ ]:
print("=" * 70)
print("CAR summary statistics")
print("=" * 70)
print(panel[["CAR[-1,+1]", "CAR[-2,+2]", "CAR[0,+1]", "CAR[0,+5]", "CAR[0,+20]",
             "|CAR|[-1,+1]"]].describe())

print(f"\n{'=' * 70}\nExclusions\n{'=' * 70}")
print(f"Original US firm-years: {len(df_us)}")
print(f"Excluded for <{MIN_ESTIMATION_OBS} estimation obs: {len(excluded_thin_estimation)}")
for company, fy, n in excluded_thin_estimation:
    print(f"  {company} FY{fy}: {n} obs")
print(f"Final panel: {len(panel)}")

# ------------------------------------------------------------
# GENERAL exclusion-accounting check, all 7 companies, not just Chubb --
# this is what actually caught the FY2023/PRICE_END bug (every company
# was silently missing exactly FY2023, not something the Chubb-specific
# check below would have caught on its own). Re-run every time this
# notebook runs, not just after a known bug, since a systematic gap like
# this produces no error/warning on its own -- rows just don't show up.
# ------------------------------------------------------------
print(f"\n{'=' * 70}\nFull exclusion accounting -- every company, every fiscal year\n{'=' * 70}")
expected_pairs = set(zip(df_us["Company"], df_us["Fiscal Year"]))
actual_pairs = set(zip(panel["Company"], panel["Fiscal Year"]))
missing_pairs = sorted(expected_pairs - actual_pairs)
unexpected_pairs = sorted(actual_pairs - expected_pairs)

print(f"Expected firm-years: {len(expected_pairs)}")
print(f"Present in final panel: {len(actual_pairs)}")

if missing_pairs:
    print(f"\nMISSING firm-years ({len(missing_pairs)}):")
    missing_by_company = {}
    for company, fy in missing_pairs:
        missing_by_company.setdefault(company, []).append(fy)
    for company, fys in missing_by_company.items():
        print(f"  {company}: {sorted(fys)}")
    # Flag whether the gap is isolated to one company (e.g. the known
    # Chubb/ACE gap) or systematic across companies (e.g. the PRICE_END
    # bug this replaced) -- these need different explanations.
    companies_affected = len(missing_by_company)
    if companies_affected > 1:
        common_fys = set.intersection(*[set(fys) for fys in missing_by_company.values()])
        if common_fys:
            print(f"\n  SYSTEMATIC: {companies_affected} companies all missing fiscal year(s) "
                  f"{sorted(common_fys)} -- check PRICE_END / event-date coverage, not a "
                  f"single-company data issue.")
else:
    print("\nNo missing firm-years.")

if unexpected_pairs:
    print(f"\nUNEXPECTED firm-years in panel not in source data ({len(unexpected_pairs)}): "
          f"{unexpected_pairs} -- investigate, this should not happen.")

assert len(panel) == 84 or (len(panel) == 80 and CHUBB_STRATEGY == "cb_only_with_gap"), (
    f"Final panel has {len(panel)} rows -- expected 84 (full) or 80 (with the known, reported "
    f"Chubb FY2012-2015 gap). Anything else means an unexplained loss -- STOP and investigate "
    f"before trusting this panel."
)
if len(panel) == 84:
    print("\nCONFIRMED: 84/84 firm-years present, no gaps.")
else:
    print(f"\nCONFIRMED: {len(panel)}/84 firm-years present, with the expected and reported "
          f"Chubb FY2012-2015 gap (CHUBB_STRATEGY={CHUBB_STRATEGY!r}) -- no OTHER unexplained gap.")

print(f"\n{'=' * 70}\nChubb / ACE ticker transition\n{'=' * 70}")
print(f"CHUBB_STRATEGY = {CHUBB_STRATEGY!r}")
chubb_panel = panel[panel["Company"] == "Chubb"]
chubb_fiscal_years_in_panel = sorted(chubb_panel["Fiscal Year"].tolist())
chubb_fiscal_years_expected = sorted(df_us[df_us["Company"] == "Chubb"]["Fiscal Year"].tolist())
chubb_missing = sorted(set(chubb_fiscal_years_expected) - set(chubb_fiscal_years_in_panel))
print(f"Chubb fiscal years in final panel: {chubb_fiscal_years_in_panel}")
if chubb_missing:
    print(f"Chubb fiscal years MISSING from final panel: {chubb_missing} "
          f"({'expected -- yfinance has no ACE-era price data, see Step 2' if CHUBB_STRATEGY == 'cb_only_with_gap' else 'unexpected, investigate'})")
else:
    print("All 12 Chubb fiscal years present.")
print(chubb_panel[["Fiscal Year", "Event Date (adjusted)"]].to_string(index=False))
if CHUBB_STRATEGY == "splice":
    print("\nSee Step 4's splice-point print for the exact ACE-end / CB-start dates used, and "
          "whether the gap-check warning fired.")
elif CHUBB_STRATEGY == "cb_only_with_gap":
    print("\nNeither CB nor ACE covers Chubb's pre-2016 history via yfinance (confirmed in "
          "Step 2) -- Chubb FY2012-2015 are excluded from this panel, not spliced or imputed. "
          "If those 4 firm-years are needed, pre-2016 ACE Limited prices would need to be "
          "sourced from a different data provider.")

print(f"\n{'=' * 70}\nSpot-check: unadjusted stock splits\n{'=' * 70}")
print("Flags any single-day return whose magnitude exceeds 20% for any of the 7 firms -- a very "
      "large one-day move that isn't obviously tied to a known market-wide shock is the classic "
      "signature of an unadjusted split slipping through (auto_adjust=True should prevent this, "
      "but checked explicitly rather than assumed).")
for company, ret in company_returns.items():
    extreme = ret[ret.abs() > 0.20]
    if len(extreme) > 0:
        print(f"\n{company}: {len(extreme)} day(s) with |return| > 20%:")
        print(extreme.to_string())
    else:
        print(f"{company}: none")


---
# Part 2 — CAR ~ Specificity + Sentiment (Extension)

Extends the Part 1 event-study panel with CEO-letter specificity and sentiment measures, and
tests whether they're associated with abnormal returns around the publication event. All data
loaded directly from Drive (same convention as the rest of this project) -- no manual uploads.

**`Specificity_{i,t}`** and **`ESGSentiment_{i,t}`** use the same **Corporate + tier-1-plus
restriction** as `Specificity_mean`/`ESGSentiment_Corporate` elsewhere in this project (confirmed
choice, not a default). **`Sentiment_{i,t}`** (overall, not ESG-restricted) uses ALL sentences
from the FinBERT file, deliberately not ESG-filtered -- it's meant to capture the letter's general
tone, not just its ESG content.

Kept in the same notebook as Part 1 (not a separate file) since it needs the just-computed
event-study panel directly. Interpretation is deliberately conservative throughout: *associated
with*, never *causes*.

## Step 6 — Merge event-study panel with specificity and sentiment data

Three sources, all from Drive:
1. `ceo_letter_event_study_panel_US.csv` (Part 1's output, 84 firm-years)
2. `df_ar_ceo_sentences_esg_combined_specificity.csv` (sentence-level, ESG-flagged sentences only,
   already carries `is_corporate_tier1plus`/`tier_if_corp_t1p`/etc.)
3. `df_ar_ceo_sentences_finbert_senti.csv` (sentence-level, ALL sentences, for overall sentiment)

Merge key: `Company` (harmonized -- the event-study panel uses the full legal names from
`Combined_Insurer_Publication_Dates.xlsx`, the CEO-letter files use the project's existing short
names) + `Fiscal Year` / `Year`. Reports matched observations, missing years, and duplicates
before building anything on top of this merge.

In [ ]:
# ============================================================
# Step 6.1 -- Load all three sources from Drive
# ============================================================
event_panel_path = "/content/drive/MyDrive/phd/Data/ceo_letter_event_study_panel_US.csv"
specificity_path = "/content/drive/MyDrive/phd/Data/df_ar_ceo_sentences_esg_combined_specificity.csv"
finbert_path = "/content/drive/MyDrive/phd/Data/df_ar_ceo_sentences_finbert_senti.csv"

event_panel = pd.read_csv(event_panel_path)
event_panel["Event Date (adjusted)"] = pd.to_datetime(event_panel["Event Date (adjusted)"])
print(f"Event-study panel: {event_panel.shape}")

df_specificity = pd.read_csv(specificity_path)
print(f"Specificity (sentence-level, ESG-flagged only): {df_specificity.shape}")

df_finbert = pd.read_csv(finbert_path)
print(f"FinBERT sentiment (sentence-level, ALL sentences): {df_finbert.shape}")

# ------------------------------------------------------------
# Company-name harmonization. The event-study panel uses the full legal
# names from Combined_Insurer_Publication_Dates.xlsx; the CEO-letter
# pipeline files use this project's existing short names (same raw
# names df_ar_cluster.csv itself is built from, before that notebook's
# own name_fixes remapping). Printed explicitly below rather than
# assumed silently correct -- if a name doesn't match, it will show up
# as 0 matched rows for that company and needs fixing here.
# ------------------------------------------------------------
print("\nEvent-study panel company names:", sorted(event_panel["Company"].unique()))
print("Specificity file company names:", sorted(df_specificity["Company Name"].unique()))
print("FinBERT file company names:", sorted(df_finbert["Company Name"].unique()))

SHORT_TO_LONG_NAME = {
    "AIG": "American International Group (AIG)",
    "Chubb": "Chubb",
    "MET": "MetLife, Inc.",
    "MetLife": "MetLife, Inc.",
    "Prudential Financials": "Prudential Financial, Inc.",
    "Prudentials": "Prudential Financial, Inc.",
    "Allstate": "The Allstate Corporation",
    "Progressive": "The Progressive Corporation",
    "Travelers": "The Travelers Companies, Inc.",
}

df_specificity["Company"] = df_specificity["Company Name"].map(SHORT_TO_LONG_NAME)
df_finbert["Company"] = df_finbert["Company Name"].map(SHORT_TO_LONG_NAME)

unmapped_spec = df_specificity[df_specificity["Company"].isna()]["Company Name"].unique()
unmapped_finbert = df_finbert[df_finbert["Company"].isna()]["Company Name"].unique()
print(f"\nUnmapped company names in specificity file (not in our 7-company mapping -- expected, "
      f"this file covers the full 22-firm panel): {sorted(unmapped_spec)}")
print(f"Unmapped company names in FinBERT file (same expectation): {sorted(unmapped_finbert)}")

# Restrict both sentence-level files to the 7 US companies actually in
# the event study, using the harmonized name -- and confirm all 7 of
# THOSE (not the broader 22-firm panel) mapped successfully.
us_companies = set(event_panel["Company"].unique())
df_specificity_us = df_specificity[df_specificity["Company"].isin(us_companies)].copy()
df_finbert_us = df_finbert[df_finbert["Company"].isin(us_companies)].copy()

mapped_companies_spec = set(df_specificity_us["Company"].unique())
mapped_companies_finbert = set(df_finbert_us["Company"].unique())
print(f"\nUS companies with NO rows at all in specificity file after mapping: "
      f"{sorted(us_companies - mapped_companies_spec)} (should be empty)")
print(f"US companies with NO rows at all in FinBERT file after mapping: "
      f"{sorted(us_companies - mapped_companies_finbert)} (should be empty)")


In [ ]:
# ============================================================
# Step 6.2 -- Firm-year aggregation: Specificity_{i,t} and
# ESGSentiment_{i,t} (Corporate + tier-1-plus restricted, matching
# Specificity_mean / ESGSentiment_Corporate elsewhere in this project).
# Mirrors the exact same aggregation logic already used there --
# is_corporate_tier1plus / tier_if_corp_t1p / is_corp_t1p_positive /
# is_corp_t1p_negative are already-computed columns in this file, not
# re-derived differently here.
# ============================================================
spec_fy = (
    df_specificity_us
    .groupby(["Company", "Year"])
    .agg(
        n_corporate_tier1plus=("is_corporate_tier1plus", "sum"),
        Specificity=("tier_if_corp_t1p", "mean"),
        n_corp_t1p_positive=("is_corp_t1p_positive", "sum"),
        n_corp_t1p_negative=("is_corp_t1p_negative", "sum"),
    )
    .reset_index()
)
spec_fy["ESGSentiment"] = np.where(
    spec_fy["n_corporate_tier1plus"] > 0,
    (spec_fy["n_corp_t1p_positive"] - spec_fy["n_corp_t1p_negative"]) / spec_fy["n_corporate_tier1plus"],
    np.nan,
)
spec_fy = spec_fy.rename(columns={"Year": "Fiscal Year"})
print(f"Specificity/ESGSentiment firm-years built: {len(spec_fy)}")
print(f"Firm-years with zero Corporate+tier1+ sentences (Specificity/ESGSentiment undefined, "
      f"NOT imputed): {(spec_fy['n_corporate_tier1plus'] == 0).sum()}")
display(spec_fy.head())

# ============================================================
# Step 6.3 -- Firm-year aggregation: Sentiment_{i,t} (overall, ALL
# sentences, not ESG-restricted). df_ar_ceo_sentences_finbert_senti.csv
# carries sent_label per sentence regardless of ESG relevance.
# ============================================================
df_finbert_us["sent_label"] = df_finbert_us["sent_label"].astype(str).str.lower().str.strip()
df_finbert_us["is_positive"] = (df_finbert_us["sent_label"] == "positive").astype(int)
df_finbert_us["is_negative"] = (df_finbert_us["sent_label"] == "negative").astype(int)

senti_fy = (
    df_finbert_us
    .groupby(["Company", "Year"])
    .agg(
        total_sentences=("Sentence", "count"),
        n_positive=("is_positive", "sum"),
        n_negative=("is_negative", "sum"),
    )
    .reset_index()
)
senti_fy["Sentiment"] = (senti_fy["n_positive"] - senti_fy["n_negative"]) / senti_fy["total_sentences"]
senti_fy = senti_fy.rename(columns={"Year": "Fiscal Year"})
print(f"\nOverall sentiment firm-years built: {len(senti_fy)}")
display(senti_fy.head())


In [ ]:
# ============================================================
# Step 6.4 -- Merge everything into the analysis panel, and report
# matched/missing/duplicates explicitly before building anything else.
# ============================================================
analysis = event_panel.merge(
    spec_fy[["Company", "Fiscal Year", "n_corporate_tier1plus", "Specificity", "ESGSentiment"]],
    on=["Company", "Fiscal Year"], how="left"
)
analysis = analysis.merge(
    senti_fy[["Company", "Fiscal Year", "total_sentences", "Sentiment"]],
    on=["Company", "Fiscal Year"], how="left"
)

print(f"Event-study panel rows: {len(event_panel)}")
print(f"Analysis panel rows after merge: {len(analysis)} (should still be 84 -- a left merge "
      f"should not change row count)")
assert len(analysis) == len(event_panel), "Row count changed after merge -- STOP, check for duplicate keys."

n_missing_specificity = analysis["Specificity"].isna().sum()
n_missing_esgsentiment = analysis["ESGSentiment"].isna().sum()
n_missing_sentiment = analysis["Sentiment"].isna().sum()
print(f"\nMissing Specificity (zero Corporate+tier1+ sentences that firm-year): {n_missing_specificity}")
print(f"Missing ESGSentiment (same reason): {n_missing_esgsentiment}")
print(f"Missing overall Sentiment (firm-year not found in FinBERT file at all): {n_missing_sentiment}")

if n_missing_specificity > 0:
    print("\nFirm-years missing Specificity:")
    print(analysis.loc[analysis["Specificity"].isna(), ["Company", "Fiscal Year"]].to_string(index=False))
if n_missing_sentiment > 0:
    print("\nFirm-years missing overall Sentiment:")
    print(analysis.loc[analysis["Sentiment"].isna(), ["Company", "Fiscal Year"]].to_string(index=False))

# Duplicate check -- should be impossible given event_panel is one row
# per (Company, Fiscal Year) and both aggregations are grouped the same
# way, but checked explicitly rather than assumed.
dup_check = analysis.duplicated(subset=["Company", "Fiscal Year"]).sum()
print(f"\nDuplicate (Company, Fiscal Year) rows in analysis panel: {dup_check} (should be 0)")

print(f"\nMatched (non-missing Specificity AND Sentiment) firm-years for regression: "
      f"{((~analysis['Specificity'].isna()) & (~analysis['Sentiment'].isna())).sum()} of {len(analysis)}")


### Step 6.5 — Diagnose the missing firm-years (concentrated, not scattered)

22 of 84 firm-years are missing `Specificity`, heavily concentrated (MetLife missing 11 of 12
years; Progressive missing 6; Prudential missing its 3 earliest years) -- not the thin, scattered
pattern a genuine "this company just didn't discuss ESG that year" explanation would produce. The
missing years also DIFFER between `Specificity` (22 firm-years) and `Sentiment` (19 firm-years),
which rules out a clean company-name-mapping failure (that would drop a company completely and
identically from both).

This checks, for exactly the missing (Company, Fiscal Year) pairs, whether the underlying
sentence-level files have: (a) no rows at all for that company-year (a source-corpus gap -- an
annual report/CEO letter that was never captured, consistent with the ALREADY-KNOWN gap-years
documented elsewhere in this project, e.g. Progressive 2020, Power Corp Canada 2022, Talanx 2020),
(b) rows present but zero ESG-flagged/Corporate+tier1+ sentences (a genuine content finding, not a
bug), or (c) something else entirely (a real remaining bug).

In [ ]:
# ============================================================
# Step 6.5 -- Row-count diagnostic for the missing firm-years
# ============================================================
missing_specificity_pairs = analysis.loc[analysis["Specificity"].isna(), ["Company", "Fiscal Year"]]
missing_sentiment_pairs = analysis.loc[analysis["Sentiment"].isna(), ["Company", "Fiscal Year"]]

print("=" * 100)
print(f"{'Company':<32} {'FY':>5}  {'spec_us rows':>13}  {'spec_us ESG-tier1+':>19}  "
      f"{'finbert_us rows':>16}  cause")
print("=" * 100)
for _, row in missing_specificity_pairs.iterrows():
    company, fy = row["Company"], row["Fiscal Year"]
    spec_rows = df_specificity_us[(df_specificity_us["Company"] == company) & (df_specificity_us["Year"] == fy)]
    n_spec_rows = len(spec_rows)
    n_tier1plus = spec_rows["is_corporate_tier1plus"].sum() if n_spec_rows else 0
    finbert_rows = df_finbert_us[(df_finbert_us["Company"] == company) & (df_finbert_us["Year"] == fy)]
    n_finbert_rows = len(finbert_rows)

    if n_spec_rows == 0 and n_finbert_rows == 0:
        cause = "SOURCE GAP: no sentences at all for this company-year in either file"
    elif n_spec_rows == 0 and n_finbert_rows > 0:
        cause = "MISMATCH: FinBERT has rows, specificity file does not -- investigate"
    elif n_spec_rows > 0 and n_tier1plus == 0:
        cause = "genuine: ESG sentences exist but none are Corporate+tier1+"
    else:
        cause = "UNEXPECTED: has tier1+ sentences but still missing -- investigate merge logic"

    print(f"{company:<32} {fy:>5}  {n_spec_rows:>13}  {n_tier1plus:>19}  {n_finbert_rows:>16}  {cause}")

print(f"\n{'=' * 70}\nFirm-years missing Sentiment specifically (not overlapping the above)\n{'=' * 70}")
sentiment_only_missing = missing_sentiment_pairs.merge(
    missing_specificity_pairs, on=["Company", "Fiscal Year"], how="left", indicator=True
)
sentiment_only_missing = sentiment_only_missing[sentiment_only_missing["_merge"] == "left_only"]
print(f"Firm-years missing Sentiment but NOT Specificity: {len(sentiment_only_missing)}")
for _, row in sentiment_only_missing.iterrows():
    company, fy = row["Company"], row["Fiscal Year"]
    finbert_rows = df_finbert_us[(df_finbert_us["Company"] == company) & (df_finbert_us["Year"] == fy)]
    print(f"  {company} FY{fy}: {len(finbert_rows)} FinBERT rows")

# ------------------------------------------------------------
# Also directly check: does the RAW (unfiltered-to-US) specificity/
# FinBERT file have any row at all under a DIFFERENT spelling for
# these company-years -- i.e. did SHORT_TO_LONG_NAME miss a spelling
# variant used only in certain years?
# ------------------------------------------------------------
print(f"\n{'=' * 70}\nAll distinct raw \'Company Name\' spellings ever used for MetLife/Prudential/"
      f"Progressive in the two source files (checking for a year-specific spelling variant)\n{'=' * 70}")
for keyword in ["Met", "Prudential", "Progressive"]:
    spec_variants = df_specificity[df_specificity["Company Name"].str.contains(keyword, case=False, na=False)]["Company Name"].unique()
    finbert_variants = df_finbert[df_finbert["Company Name"].str.contains(keyword, case=False, na=False)]["Company Name"].unique()
    print(f"\n\'{keyword}\' -- specificity file: {sorted(spec_variants)}")
    print(f"\'{keyword}\' -- FinBERT file: {sorted(finbert_variants)}")


## Step 7 — Complete-case sample, descriptive statistics, correlations

Regression sample = firm-years with both `Specificity`/`ESGSentiment` and `Sentiment` defined
(non-imputed; a firm-year with zero Corporate+tier1+ sentences is dropped, not filled with 0 or
a mean). Tables 1 and 2 as specified.

In [ ]:
# ============================================================
# Step 7.1 -- Complete-case regression sample
# ============================================================
reg_cols = ["Specificity", "ESGSentiment", "Sentiment", "CAR[-1,+1]", "|CAR|[-1,+1]"]
reg_sample = analysis.dropna(subset=reg_cols).copy()
print(f"Complete-case regression sample: {len(reg_sample)} of {len(analysis)} firm-years")
print(f"Dropped: {len(analysis) - len(reg_sample)} "
      f"({analysis[~analysis.index.isin(reg_sample.index)][['Company', 'Fiscal Year']].to_string(index=False) if len(analysis) > len(reg_sample) else 'none'})")

# ============================================================
# Step 7.2 -- Table 1: descriptive statistics
# ============================================================
table1_cols = ["CAR[-1,+1]", "|CAR|[-1,+1]", "CAR[-2,+2]", "CAR[0,+1]", "Specificity", "ESGSentiment", "Sentiment"]
table1_cols = [c for c in table1_cols if c in reg_sample.columns]
table1 = reg_sample[table1_cols].agg(["mean", "std", "min", "max", "count"]).T
print("Table 1 -- Descriptive statistics")
display(table1.round(4))

# ============================================================
# Step 7.3 -- Table 2: correlation matrix
# ============================================================
table2_cols = ["CAR[-1,+1]", "|CAR|[-1,+1]", "Specificity", "ESGSentiment", "Sentiment"]
table2 = reg_sample[table2_cols].corr()
print("\nTable 2 -- Correlation matrix")
display(table2.round(3))


## Step 8 — Regression models

Four model specifications (Specificity only; Sentiment only; Specificity + Sentiment; + interaction),
each under three fixed-effects specifications (pooled OLS; firm FE; firm + year FE). Primary DV
`CAR[-1,+1]` and `|CAR|[-1,+1]`; robustness windows (`CAR[-2,+2]`, `CAR[0,+1]`, `CAR[0,+5]`,
`CAR[0,+20]` if present) reported separately below, not repeated across all 4x3 combinations to
keep the primary table readable.

**Small-sample caveat, stated once rather than repeated on every table below:** N is at most 84
(likely fewer after the complete-case restriction), with only 7 firms. Firm+year FE with 7 firms
and up to 12 years uses a large share of the available degrees of freedom -- coefficients and SEs
from Specification C should be read as exploratory, not as precision estimates on the scale a
larger panel would give. Firm-clustered SEs with only 7 clusters are below the ~30-50 cluster
rule of thumb for reliable cluster-robust asymptotics; reported anyway (as requested) alongside
HC3, with that caveat attached rather than silently omitted.

In [ ]:
# ============================================================
# Step 8.1 -- Fit all Model x Specification combinations for a given DV
# ============================================================
import statsmodels.formula.api as smf

MODEL_FORMULAS = {
    "Model 1 (Specificity only)": "Specificity",
    "Model 2 (Sentiment only)": "Sentiment",
    "Model 3 (Specificity + Sentiment)": "Specificity + Sentiment",
    "Model 4 (+ interaction)": "Specificity + Sentiment + Specificity:Sentiment",
}

FE_SPECS = {
    "A: Pooled OLS": "",
    "B: Firm FE": " + C(Company)",
    "C: Firm + Year FE": " + C(Company) + C(Q(\'Fiscal Year\'))",
}

def fit_all_models(df, dv, cov_type="HC3", cov_kwds=None):
    out = {}
    for model_label, rhs in MODEL_FORMULAS.items():
        for fe_label, fe_terms in FE_SPECS.items():
            formula = f"Q(\'{dv}\') ~ {rhs}{fe_terms}"
            try:
                kwargs = {"cov_type": cov_type}
                if cov_kwds is not None:
                    kwargs["cov_kwds"] = cov_kwds
                m = smf.ols(formula=formula, data=df).fit(**kwargs)
                out[(model_label, fe_label)] = m
            except Exception as e:
                print(f"FAILED: {model_label} / {fe_label} / {dv}: {e}")
    return out

models_car_hc3 = fit_all_models(reg_sample, "CAR[-1,+1]", cov_type="HC3")
models_abscar_hc3 = fit_all_models(reg_sample, "|CAR|[-1,+1]", cov_type="HC3")

models_car_cluster = fit_all_models(
    reg_sample, "CAR[-1,+1]", cov_type="cluster", cov_kwds={"groups": reg_sample["Company"]}
)
models_abscar_cluster = fit_all_models(
    reg_sample, "|CAR|[-1,+1]", cov_type="cluster", cov_kwds={"groups": reg_sample["Company"]}
)

print(f"Fitted {len(models_car_hc3)} model x FE-spec combinations for CAR[-1,+1] (HC3)")
print(f"Fitted {len(models_abscar_hc3)} model x FE-spec combinations for |CAR|[-1,+1] (HC3)")


In [ ]:
# ============================================================
# Step 8.2 -- Table 3: main results, Model 3 and 4, HC3 and cluster SEs,
# all three FE specs, both DVs.
# ============================================================
def summarize_model(m, terms):
    rows = []
    for term in terms:
        if term not in m.params.index:
            continue
        rows.append({
            "term": term, "coef": m.params[term], "se": m.bse[term], "p": m.pvalues[term],
            "N": int(m.nobs), "R2": m.rsquared,
        })
    return rows

table3_rows = []
key_terms = ["Specificity", "Sentiment", "Specificity:Sentiment"]

for dv_label, models_hc3, models_cluster in [
    ("CAR[-1,+1]", models_car_hc3, models_car_cluster),
    ("|CAR|[-1,+1]", models_abscar_hc3, models_abscar_cluster),
]:
    for model_label in ["Model 3 (Specificity + Sentiment)", "Model 4 (+ interaction)"]:
        for fe_label in FE_SPECS:
            key = (model_label, fe_label)
            if key not in models_hc3:
                continue
            for row in summarize_model(models_hc3[key], key_terms):
                row.update({"dv": dv_label, "model": model_label, "fe": fe_label, "se_type": "HC3"})
                table3_rows.append(row)
            for row in summarize_model(models_cluster[key], key_terms):
                row.update({"dv": dv_label, "model": model_label, "fe": fe_label, "se_type": "cluster(firm)"})
                table3_rows.append(row)

table3 = pd.DataFrame(table3_rows)
table3["sig"] = table3["p"].apply(lambda p: "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else "")

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)
print("Table 3 -- Main regression results (Models 3-4, all FE specs, HC3 and cluster SEs)")
display(table3[["dv", "model", "fe", "se_type", "term", "coef", "se", "p", "sig", "N", "R2"]].round(4))

# ------------------------------------------------------------
# Models 1-2 (single-regressor) reported separately, more compactly --
# these establish the baseline before Models 3-4's combined/interaction
# story.
# ------------------------------------------------------------
table_m1m2_rows = []
for dv_label, models_hc3 in [("CAR[-1,+1]", models_car_hc3), ("|CAR|[-1,+1]", models_abscar_hc3)]:
    for model_label in ["Model 1 (Specificity only)", "Model 2 (Sentiment only)"]:
        for fe_label in FE_SPECS:
            key = (model_label, fe_label)
            if key not in models_hc3:
                continue
            for row in summarize_model(models_hc3[key], key_terms):
                row.update({"dv": dv_label, "model": model_label, "fe": fe_label})
                table_m1m2_rows.append(row)
table_m1m2 = pd.DataFrame(table_m1m2_rows)
table_m1m2["sig"] = table_m1m2["p"].apply(lambda p: "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else "")
print("\nModels 1-2 (single-regressor baselines, HC3 SEs)")
display(table_m1m2[["dv", "model", "fe", "term", "coef", "se", "p", "sig", "N", "R2"]].round(4))


## Step 9 — Robustness

1. Alternative CAR windows (Model 3, Specification C only, to keep this compact).
2. Influential observations: Cook's distance and studentized residuals on the primary Model 3 /
   Spec C / `CAR[-1,+1]` fit, with AIG FY2019 and any COVID-period (2020) event specifically
   called out, plus the fit re-estimated excluding whatever the data actually flags (not
   pre-assumed to be those specific points).
3. Wild cluster bootstrap: **attempted, with an explicit caveat**. With only 7 firms, cluster-
   robust inference (HC3-cluster or a full wild bootstrap alike) is below the conventional
   30-50-cluster guidance for reliable asymptotics -- reported for completeness, not because 7
   clusters make this reliable.

In [ ]:
# ============================================================
# Step 9.1 -- Alternative CAR windows, Model 3 / Spec C only
# ============================================================
alt_windows = ["CAR[-1,+1]", "CAR[-2,+2]", "CAR[0,+1]"]
if "CAR[0,+5]" in reg_sample.columns:
    alt_windows.append("CAR[0,+5]")
if "CAR[0,+20]" in reg_sample.columns:
    alt_windows.append("CAR[0,+20]")
else:
    print("NOTE: CAR[0,+5]/CAR[0,+20] not present in this panel -- re-run Part 1's Step 4/5 "
          "(already updated to compute them) and re-upload to include these two windows here.")

window_rows = []
for dv in alt_windows:
    if reg_sample[dv].isna().all():
        continue
    sub = reg_sample.dropna(subset=["Specificity", "Sentiment", dv])
    formula = f"Q('{dv}') ~ Specificity + Sentiment + C(Company) + C(Q('Fiscal Year'))"
    m = smf.ols(formula=formula, data=sub).fit(cov_type="HC3")
    for term in ["Specificity", "Sentiment"]:
        window_rows.append({
            "window": dv, "term": term, "coef": m.params[term], "se": m.bse[term],
            "p": m.pvalues[term], "N": int(m.nobs),
        })

table_windows = pd.DataFrame(window_rows)
print("Model 3, Firm+Year FE, HC3 SEs -- across CAR windows")
display(table_windows.round(4))


In [ ]:
# ============================================================
# Step 9.2 -- Influential observations: Cook's distance on the primary
# fit (Model 3, Spec C, CAR[-1,+1], HC3).
# ============================================================
from statsmodels.stats.outliers_influence import OLSInfluence

primary_fit = models_car_hc3[("Model 3 (Specificity + Sentiment)", "C: Firm + Year FE")]
influence = OLSInfluence(primary_fit)
cooks_d = influence.cooks_distance[0]

fit_sample = reg_sample.loc[primary_fit.model.data.row_labels].copy()
fit_sample["cooks_d"] = cooks_d
fit_sample["studentized_resid"] = influence.resid_studentized_external

threshold = 4 / primary_fit.nobs
flagged = fit_sample[fit_sample["cooks_d"] > threshold].sort_values("cooks_d", ascending=False)
print(f"Cook's distance threshold (4/N): {threshold:.4f}")
print(f"Flagged observations: {len(flagged)} of {int(primary_fit.nobs)}")
display(flagged[["Company", "Fiscal Year", "CAR[-1,+1]", "Specificity", "Sentiment",
                  "cooks_d", "studentized_resid"]])

aig_2019 = fit_sample[(fit_sample["Company"].str.contains("AIG")) & (fit_sample["Fiscal Year"] == 2019)]
covid_events = fit_sample[fit_sample["Fiscal Year"] == 2020]
print("\nAIG FY2019 specifically:")
if len(aig_2019):
    print(aig_2019[["Company", "Fiscal Year", "cooks_d", "studentized_resid"]].to_string(index=False))
else:
    print("not in complete-case sample")
print("\nAll FY2020 (COVID-period) events:")
if len(covid_events):
    print(covid_events[["Company", "Fiscal Year", "cooks_d", "studentized_resid"]].to_string(index=False))
else:
    print("none in complete-case sample")

# ------------------------------------------------------------
# Re-fit excluding whatever Cook's distance actually flagged (not a
# pre-assumed set) -- report whether Specificity/Sentiment survive.
# ------------------------------------------------------------
if len(flagged) > 0:
    excl_idx = flagged.index
    sub_excl = reg_sample.drop(index=[i for i in excl_idx if i in reg_sample.index])
    m_excl = smf.ols(
        formula="Q('CAR[-1,+1]') ~ Specificity + Sentiment + C(Company) + C(Q('Fiscal Year'))",
        data=sub_excl
    ).fit(cov_type="HC3")
    print(f"\nModel 3 / Spec C refit excluding {len(flagged)} flagged observation(s): "
          f"N={int(m_excl.nobs)}")
    for term in ["Specificity", "Sentiment"]:
        print(f"  {term}: coef={m_excl.params[term]:.4f}, se={m_excl.bse[term]:.4f}, "
              f"p={m_excl.pvalues[term]:.4f}")
else:
    print("\nNo observations exceeded the Cook's distance threshold -- no exclusion re-fit needed.")


In [ ]:
# ============================================================
# Step 9.3 -- Standard errors: HC3 (already default above), firm-
# clustered (already computed above), and a pairs-cluster bootstrap
# (resampling firms with replacement) as the feasible approximation to
# a full wild cluster bootstrap here.
#
# CAVEAT, stated explicitly rather than glossed over: with only 7
# firms, NO clustering method -- analytic cluster-robust, wild
# bootstrap, or this pairs bootstrap -- has good asymptotic properties.
# The conventional guidance wants 30-50+ clusters. Reported because it
# was requested, not because 7 clusters makes any of these reliable.
# ============================================================
N_BOOTSTRAP = 2000
rng = np.random.default_rng(42)
firms = reg_sample["Company"].unique()

boot_coefs = {"Specificity": [], "Sentiment": []}
base_formula = "Q('CAR[-1,+1]') ~ Specificity + Sentiment + C(Company) + C(Q('Fiscal Year'))"

for b in range(N_BOOTSTRAP):
    sampled_firms = rng.choice(firms, size=len(firms), replace=True)
    # Re-key each resampled draw of a firm to its own dummy level, so
    # C(Company) doesn't silently merge two independent draws of the
    # same firm (which would happen if the same "Company" string
    # appeared twice with replace=True resampling).
    boot_frames = []
    for draw_i, f in enumerate(sampled_firms):
        block = reg_sample[reg_sample["Company"] == f].copy()
        block["Company"] = f"{f}__draw{draw_i}"
        boot_frames.append(block)
    boot_df = pd.concat(boot_frames, ignore_index=True)
    try:
        m_boot = smf.ols(formula=base_formula, data=boot_df).fit()
        for term in ["Specificity", "Sentiment"]:
            if term in m_boot.params.index:
                boot_coefs[term].append(m_boot.params[term])
    except Exception:
        continue

n_boot_success = len(boot_coefs["Specificity"])
print(f"Successful bootstrap draws: {n_boot_success} of {N_BOOTSTRAP} requested")

primary_model_key = ("Model 3 (Specificity + Sentiment)", "C: Firm + Year FE")
hc3_bse = models_car_hc3[primary_model_key].bse
cluster_bse = models_car_cluster[primary_model_key].bse
point_ests = models_car_hc3[primary_model_key].params

for term in ["Specificity", "Sentiment"]:
    vals = np.array(boot_coefs[term])
    if len(vals) > 10:
        se_boot = vals.std(ddof=1)
        ci_lo, ci_hi = np.percentile(vals, [2.5, 97.5])
        print(f"\n{term}: point estimate={point_ests[term]:.4f}, "
              f"pairs-cluster-bootstrap SE={se_boot:.4f}, 95% pctile CI=[{ci_lo:.4f}, {ci_hi:.4f}]")
        print(f"  (compare to HC3 SE={hc3_bse[term]:.4f}, cluster SE={cluster_bse[term]:.4f})")
    else:
        print(f"\n{term}: too few successful bootstrap draws to report a stable SE/CI.")


## Report-only summary

Findings only -- no interpretation of causality, and this analysis is explicitly exploratory
given N<=84 across 7 firms. *The analysis examines whether CEO-letter narrative characteristics
are associated with abnormal stock returns around annual-report publication events* -- not that
CEO letters cause stock-market movements.

In [ ]:
print("=" * 70)
print("Merge diagnostics")
print("=" * 70)
print(f"Event-study panel: {len(event_panel)} firm-years")
n_dropped = len(event_panel) - len(reg_sample)
print(f"Complete-case regression sample: {len(reg_sample)} firm-years "
      f"({n_dropped} dropped for missing Specificity/ESGSentiment/Sentiment)")

section_rule = "=" * 70
print(f"\n{section_rule}\nModel 3 (Specificity + Sentiment), Firm+Year FE -- primary results\n{section_rule}")
for dv_label, models_hc3, models_cluster in [
    ("CAR[-1,+1]", models_car_hc3, models_car_cluster),
    ("|CAR|[-1,+1]", models_abscar_hc3, models_abscar_cluster),
]:
    m_hc3 = models_hc3[("Model 3 (Specificity + Sentiment)", "C: Firm + Year FE")]
    m_cl = models_cluster[("Model 3 (Specificity + Sentiment)", "C: Firm + Year FE")]
    print(f"\nDV = {dv_label}, N={int(m_hc3.nobs)}, R2={m_hc3.rsquared:.4f}")
    for term in ["Specificity", "Sentiment"]:
        print(f"  {term}: coef={m_hc3.params[term]:.4f}, "
              f"HC3 se={m_hc3.bse[term]:.4f} (p={m_hc3.pvalues[term]:.4f}), "
              f"cluster se={m_cl.bse[term]:.4f} (p={m_cl.pvalues[term]:.4f})")

print(f"\n{section_rule}\nInterpretation (conservative, per instruction)\n{section_rule}")
print("The analysis examines whether CEO-letter narrative characteristics are associated with")
print("abnormal stock returns around annual-report publication events -- not that CEO letters")
print("cause stock-market movements. N is small (<=84 firm-years, 7 firms); firm-clustered and")
print("bootstrap SEs both carry the small-cluster caveat noted in Step 9.3.")


---
# Part 3 — Non-US extension: ticker verification (15 companies)

Extends the event study beyond the 7 US firms to the other 15 companies in the full 22-firm
panel (Canada, China/HK, France, Germany, Italy, Switzerland, Japan). **This cell only verifies
candidate tickers resolve on Yahoo Finance -- it does not pull price data or run any event-study
computation.** Candidate tickers below are my best-guess identifiers for each company's primary
listing; none have been empirically confirmed yet. Run this cell, inspect the output, and report
back which resolve before any price pull is attempted -- same "probe before pulling" pattern used
for the ACE/Chubb ticker resolution in Part 1.

**Two companies have a name change in the source spreadsheet, analogous to the Chubb/ACE
situation, but NOT (as far as I can tell) a ticker change** -- both are the same legal entity
before and after a rename, still listed under the same Tokyo Stock Exchange code throughout:

- Dai-ichi: `'Dai-ichi Life Insurance Company / Dai-ichi Life Holdings'` (FY2012-2015) →
  `'Dai-ichi Life Holdings'` (FY2016-2023). Candidate ticker `8750.T` for both.
- Sompo: `'NKSJ Holdings / Sompo Holdings'` (FY2012-2013) → `'Sompo Holdings'` (FY2014-2023).
  Candidate ticker `8630.T` for both.

This is an assumption, not a confirmed fact -- the probe below checks coverage across the FULL
2012-2023 span for both tickers specifically to test it, the same way Part 1 empirically tested
(rather than assumed) whether `CB` covered the pre-2016 ACE years.

In [ ]:
# ============================================================
# Non-US candidate tickers -- 15 companies. UNVERIFIED candidates;
# this cell's job is to test them, not assume they are right.
# ============================================================
NONUS_COMPANY_TICKER = {
    # Canada
    "Power Corporation of Canada": "POW.TO",
    # China / Hong Kong
    "China Life Insurance Company Limited": "2628.HK",
    "People's Insurance Company of China (PICC)": "1339.HK",
    "Ping An Insurance Group": "2318.HK",
    # France
    "AXA SA": "CS.PA",
    # Germany
    "Allianz SE": "ALV.DE",
    "Munich Re": "MUV2.DE",
    "Talanx AG": "TLX.DE",
    # Italy
    "Assicurazioni Generali": "G.MI",
    # Switzerland
    "Swiss Re": "SREN.SW",
    "Zurich Insurance Group": "ZURN.SW",
    # Japan
    "Dai-ichi Life Holdings": "8750.T",
    "Dai-ichi Life Insurance Company / Dai-ichi Life Holdings": "8750.T",
    "MS&AD Insurance Group Holdings": "8725.T",
    "Sompo Holdings": "8630.T",
    "NKSJ Holdings / Sompo Holdings": "8630.T",
    "Tokio Marine Holdings": "8766.T",
}

# Distinct tickers to probe (the two Japan renames collapse to one ticker each).
NONUS_TICKERS = sorted(set(NONUS_COMPANY_TICKER.values()))
print(f"{len(NONUS_COMPANY_TICKER)} company-name entries -> {len(NONUS_TICKERS)} distinct tickers to probe")
for t in NONUS_TICKERS:
    companies = [c for c, tk in NONUS_COMPANY_TICKER.items() if tk == t]
    print(f"  {t}: {companies}")


In [ ]:
# ============================================================
# Probe each ticker in two windows: an EARLY window (2012, the panel's
# first fiscal year) and a RECENT window (2023), non-fatally. This
# mirrors the ACE/CB pattern (probe before pulling), but checks BOTH
# ends of the range up front, since for Dai-ichi/Sompo specifically we
# need to know whether one ticker really covers the full 2012-2023 span
# under the assumed no-ticker-change reading above -- not just whether
# it resolves at all today.
# ============================================================
import yfinance as yf

EARLY_PROBE = ("2012-01-01", "2012-02-15")
RECENT_PROBE = ("2023-01-01", "2023-02-15")

probe_results = {}
for ticker in NONUS_TICKERS:
    row = {"ticker": ticker}
    for label, window in [("early_2012", EARLY_PROBE), ("recent_2023", RECENT_PROBE)]:
        start, end = window
        try:
            data = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
            row[label] = "OK" if len(data) > 0 else "EMPTY (0 rows, no error)"
        except Exception as e:
            row[label] = f"FAILED: {e}"
    probe_results[ticker] = row
    early_status = row["early_2012"]
    recent_status = row["recent_2023"]
    print(f"{ticker}: early_2012={early_status!r}, recent_2023={recent_status!r}")

probe_df = pd.DataFrame(probe_results).T
print()
display(probe_df)

# ------------------------------------------------------------
# Flag anything that needs a decision before pulling: a ticker that
# fails EITHER window (candidate is wrong / listing changed / delisted)
# or that resolves only in one window (possible real ticker change,
# same situation as ACE -> CB, needing the same kind of splice logic).
# ------------------------------------------------------------
problem_tickers = probe_df[
    probe_df["early_2012"].str.contains("FAILED|EMPTY")
    | probe_df["recent_2023"].str.contains("FAILED|EMPTY")
]
n_problem = len(problem_tickers)
n_total = len(probe_df)
print(f"\nTickers needing attention before any price pull: {n_problem} of {n_total}")
if n_problem:
    display(problem_tickers)
else:
    print("All candidate tickers resolved in both the 2012 and 2023 windows.")


In [ ]:
# ============================================================
# Benchmark check -- confirm URTH (already used for the US 7-firm
# panel) also covers back to 2012, since the non-US event windows will
# be compared against the same benchmark.
# ============================================================
try:
    urth_2012 = yf.download("URTH", start="2012-01-01", end="2012-02-15", progress=False, auto_adjust=False)
    status = "OK" if len(urth_2012) > 0 else "EMPTY"
    n_rows = len(urth_2012)
    print(f"URTH 2012 probe: {status} ({n_rows} rows)")
except Exception as e:
    print(f"URTH 2012 probe FAILED: {e}")


## Step 10 — Non-US event dates and canonical company names

Builds the 15-company non-US firm-year table from the same source file, and canonicalizes the
Dai-ichi and Sompo name-change pairs to a single `Company` label each (both are the same legal
entity, same ticker throughout -- confirmed in Part 3's probe, not assumed).

In [ ]:
# ============================================================
# Step 10.1 -- Non-US firm-year table + canonical company names
# ============================================================
df_nonus = df_dates_raw[df_dates_raw["Region"] != "United States"].copy()
df_nonus["Publication Date_raw"] = df_nonus["Publication Date"].astype(str)
# Some rows (following the user's manual re-verification of Japanese
# Integrated/Annual Report dates against primary sources, since the prior
# EDINET-derived dates for 5 Japanese companies were the wrong document --
# a regulatory securities-report filing deadline, not the actual investor
# Annual/Integrated Report publication date) are now NaN (day genuinely
# unverifiable) or a "YYYY-MM" month-only string (day not independently
# confirmed). Both are flagged, NOT dropped from df_nonus and NOT given an
# invented day -- Step 12 skips them explicitly rather than silently using
# a wrong or made-up event date.
df_nonus["Has_Day_Precision_Date"] = df_nonus["Publication Date_raw"].str.match(r"^\d{4}-\d{2}-\d{2}")
df_nonus["Publication Date"] = pd.to_datetime(df_nonus["Publication Date"], format="mixed", errors="coerce")
print(f"Non-US rows: {len(df_nonus)}")
n_no_day_precision = (~df_nonus["Has_Day_Precision_Date"]).sum()
print(f"Rows WITHOUT a verified exact publication day (NaN or month-only -- excluded from the "
      f"event-study loop in Step 12, not given an invented day): {n_no_day_precision} of {len(df_nonus)}")
if n_no_day_precision > 0:
    display(df_nonus.loc[~df_nonus["Has_Day_Precision_Date"],
                          ["Company", "Fiscal Year", "Publication Date_raw", "Note"]])
print(f"Distinct raw company-name strings: {df_nonus['Company'].nunique()}")

NONUS_CANONICAL_COMPANY = {
    "Power Corporation of Canada": "Power Corporation of Canada",
    "China Life Insurance Company Limited": "China Life Insurance Company Limited",
    "People\'s Insurance Company of China (PICC)": "People\'s Insurance Company of China (PICC)",
    "Ping An Insurance Group": "Ping An Insurance Group",
    "AXA SA": "AXA SA",
    "Allianz SE": "Allianz SE",
    "Munich Re": "Munich Re",
    "Talanx AG": "Talanx AG",
    "Assicurazioni Generali": "Assicurazioni Generali",
    "Swiss Re": "Swiss Re",
    "Zurich Insurance Group": "Zurich Insurance Group",
    "Dai-ichi Life Holdings": "Dai-ichi Life Holdings",
    "Dai-ichi Life Insurance Company / Dai-ichi Life Holdings": "Dai-ichi Life Holdings",
    "MS&AD Insurance Group Holdings": "MS&AD Insurance Group Holdings",
    "Sompo Holdings": "Sompo Holdings",
    "NKSJ Holdings / Sompo Holdings": "Sompo Holdings",
    "Tokio Marine Holdings": "Tokio Marine Holdings",
}

df_nonus["Company_canonical"] = df_nonus["Company"].map(NONUS_CANONICAL_COMPANY)
n_unmapped = df_nonus["Company_canonical"].isna().sum()
if n_unmapped > 0:
    print("UNMAPPED raw company names (STOP, fix NONUS_CANONICAL_COMPANY before continuing):")
    print(sorted(df_nonus.loc[df_nonus["Company_canonical"].isna(), "Company"].unique()))
assert n_unmapped == 0, f"{n_unmapped} non-US rows have no canonical company mapping -- STOP."

n_canonical_companies = df_nonus["Company_canonical"].nunique()
print(f"Canonical companies: {n_canonical_companies} (expected 15)")
assert n_canonical_companies == 15, f"Expected 15 canonical non-US companies, got {n_canonical_companies}"

counts_per_company = df_nonus.groupby("Company_canonical").size()
print("\nRows per canonical company (Dai-ichi and Sompo should each sum their two name-variant "
      "rows to 12):")
print(counts_per_company.to_string())
assert (counts_per_company == 12).all(), "Not every canonical company has exactly 12 fiscal years -- STOP."
print("\nCONFIRMED: 180 non-US firm-years across 15 canonical companies x 12 fiscal years.")


## Step 11 — Non-US price pull and return series

Reuses `NONUS_COMPANY_TICKER` from Part 3. No splice logic needed here (unlike Chubb/ACE) --
Part 3's probe confirmed a single ticker per canonical company covers the full 2012-2023 range,
except PICC (`1339.HK`) and Talanx (`TLX.DE`), whose actual price history starts partway through
2012 because both IPO'd that year (confirmed via their empty-early-2012 / OK-2023 probe results,
consistent with PICC's Dec-2012 HKEX listing and Talanx's Jun-2012 Frankfurt listing) -- not a
splice case, just a real, later start date that Step 12's existing
`MIN_ESTIMATION_OBS` check will handle by excluding their FY2012 (and possibly FY2013) event
naturally, the same way any other data-gap firm-year already gets excluded.

In [ ]:
# ============================================================
# Step 11.1 -- Pull daily price data for the 15 non-US tickers
# (URTH already pulled/available as `benchmark_returns` from Step 3-4).
# ============================================================
raw_prices_nonus = {}
for ticker in NONUS_TICKERS:
    print(f"Pulling {ticker}...")
    data = yf.download(ticker, start=PRICE_START, end=PRICE_END, progress=False, auto_adjust=True)
    raw_prices_nonus[ticker] = data
    if len(data) > 0:
        print(f"  {len(data)} rows, {data.index.min()} to {data.index.max()}")
    else:
        print("  NO DATA -- STOP, investigate before continuing (Part 3 probe said this resolved).")

raw_prices_nonus_dir = "/content/drive/MyDrive/phd/Data/event_study_raw_prices_nonus"
os.makedirs(raw_prices_nonus_dir, exist_ok=True)
for ticker, data in raw_prices_nonus.items():
    out_path = os.path.join(raw_prices_nonus_dir, f"{ticker}.csv")
    data.to_csv(out_path)
print(f"\nAll raw pulls saved to: {raw_prices_nonus_dir}")


In [ ]:
# ============================================================
# Step 11.2 -- Build one return series per canonical company. No
# splice needed (see markdown above) -- just each company\'s single
# ticker\'s own pct_change.
# ============================================================
def get_adj_close_from(price_dict, ticker):
    close = price_dict[ticker]["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.squeeze("columns")
    return close.dropna()

TICKER_BY_CANONICAL = {}
for raw_name, canonical in NONUS_CANONICAL_COMPANY.items():
    TICKER_BY_CANONICAL[canonical] = NONUS_COMPANY_TICKER[raw_name]

company_returns_nonus = {}
for canonical, ticker in TICKER_BY_CANONICAL.items():
    close = get_adj_close_from(raw_prices_nonus, ticker)
    returns = close.pct_change().dropna()
    returns.name = canonical
    company_returns_nonus[canonical] = returns
    print(f"{canonical} ({ticker}): {len(returns)} return observations, "
          f"{returns.index.min()} to {returns.index.max()}")


## Step 12 — Non-US market-model event study (calendar-mismatch aware)

Same market-model mechanics as Step 4 (estimation window `[-250,-30]`, event windows summed as
abnormal-return CAR, `MIN_ESTIMATION_OBS = 100`), but **each non-US firm trades on its own local
exchange calendar (Tokyo, Hong Kong, Frankfurt, Paris, Milan, Zurich, Toronto), while the
benchmark `URTH` trades on the US calendar** -- so a day the firm trades and the benchmark doesn't
(a local holiday that isn't a US holiday, or vice versa) is a real, expected mismatch, far more
frequent than for the 7 US companies (which share URTH's own calendar almost exactly). Step 4's
original code silently dropped these days from the abnormal-return sum via `.sum(skipna=True)` --
here that drop is counted and reported explicitly per firm-year, for both the estimation window
and the primary event window, rather than left invisible.

In [ ]:
# ============================================================
# Step 12.1 -- Event-study loop, non-US firms, with explicit
# calendar-mismatch counting/reporting.
# ============================================================
results_nonus = []
n_shifted_nonus = 0
excluded_thin_estimation_nonus = []
excluded_no_day_precision_nonus = []

for _, row in df_nonus.iterrows():
    company = row["Company_canonical"]
    fiscal_year = row["Fiscal Year"]
    raw_event_date = row["Publication Date"]

    if not row["Has_Day_Precision_Date"]:
        excluded_no_day_precision_nonus.append((company, fiscal_year, row["Publication Date_raw"]))
        continue

    firm_ret = company_returns_nonus[company]
    firm_calendar = firm_ret.index

    if raw_event_date in firm_calendar:
        event_date = raw_event_date
    else:
        event_date = next_trading_day(raw_event_date, firm_calendar)
        n_shifted_nonus += 1
        if event_date is None:
            print(f"SKIPPING {company} FY{fiscal_year}: no trading day found on/after "
                  f"{raw_event_date} in the available price history.")
            continue

    # ---- estimation window ----
    est_start = trading_day_offset(event_date, firm_calendar, ESTIMATION_WINDOW[0])
    est_end = trading_day_offset(event_date, firm_calendar, ESTIMATION_WINDOW[1])
    if est_start is None or est_end is None:
        print(f"SKIPPING {company} FY{fiscal_year}: estimation window falls outside available "
              f"price history.")
        continue

    est_mask = (firm_ret.index >= est_start) & (firm_ret.index <= est_end)
    est_firm = firm_ret[est_mask]
    est_bench_raw = benchmark_returns.reindex(est_firm.index)
    est_calendar_mismatch_n = int(est_bench_raw.isna().sum())
    est_bench = est_bench_raw.dropna()
    est_common = est_firm.index.intersection(est_bench.index)
    est_firm = est_firm.loc[est_common]
    est_bench = benchmark_returns.loc[est_common]

    n_est = len(est_common)
    if n_est < MIN_ESTIMATION_OBS:
        excluded_thin_estimation_nonus.append((company, fiscal_year, n_est))
        continue

    X = np.column_stack([np.ones(n_est), est_bench.values])
    coefs, _, _, _ = np.linalg.lstsq(X, est_firm.values, rcond=None)
    alpha, beta = coefs[0], coefs[1]

    # ---- event windows, with calendar-mismatch counted per window ----
    car_values = {}
    car_calendar_gaps = {}
    for win_name, (lo, hi) in EVENT_WINDOWS.items():
        win_start = trading_day_offset(event_date, firm_calendar, lo)
        win_end = trading_day_offset(event_date, firm_calendar, hi)
        if win_start is None or win_end is None:
            car_values[win_name] = np.nan
            car_calendar_gaps[win_name] = np.nan
            continue
        win_mask = (firm_ret.index >= win_start) & (firm_ret.index <= win_end)
        win_firm = firm_ret[win_mask]
        win_bench = benchmark_returns.reindex(win_firm.index)
        gap_n = int(win_bench.isna().sum())
        car_calendar_gaps[win_name] = gap_n
        abnormal = win_firm - (alpha + beta * win_bench)
        car_values[win_name] = abnormal.sum()  # skipna=True by default -- gap_n above makes this explicit

    results_nonus.append({
        "Company": company,
        "Fiscal Year": fiscal_year,
        "Event Date (raw)": raw_event_date,
        "Event Date (adjusted)": event_date,
        "Shifted": event_date != raw_event_date,
        "Estimation N": n_est,
        "Est_Calendar_Mismatch_N": est_calendar_mismatch_n,
        "Alpha": alpha,
        "Beta": beta,
        "CAR[-1,+1]": car_values["primary_-1_+1"],
        "CAR[-1,+1]_Calendar_Gaps": car_calendar_gaps["primary_-1_+1"],
        "CAR[-2,+2]": car_values["robustness_-2_+2"],
        "CAR[0,+1]": car_values["robustness_0_+1"],
        "CAR[0,+5]": car_values["robustness_0_+5"],
        "CAR[0,+20]": car_values["robustness_0_+20"],
        "|CAR|[-1,+1]": abs(car_values["primary_-1_+1"]),
    })

print(f"Publication dates shifted to next trading day: {n_shifted_nonus} of {len(df_nonus)}")
print(f"\nFirm-years excluded for <{MIN_ESTIMATION_OBS} estimation-window observations: "
      f"{len(excluded_thin_estimation_nonus)}")
for company, fy, n in excluded_thin_estimation_nonus:
    print(f"  {company} FY{fy}: only {n} estimation-window observations")

print(f"\nFirm-years excluded for no verified exact publication day (NaN or month-only in the "
      f"source spreadsheet -- see Step 10.1): {len(excluded_no_day_precision_nonus)}")
for company, fy, raw_val in excluded_no_day_precision_nonus:
    print(f"  {company} FY{fy}: Publication Date = {raw_val!r}")


In [ ]:
# ============================================================
# Step 12.2 -- Calendar-mismatch report. Explicit, not folded silently
# into the CAR sums above.
# ============================================================
panel_nonus = pd.DataFrame(results_nonus)
panel_nonus = panel_nonus.sort_values(["Company", "Fiscal Year"]).reset_index(drop=True)

print(f"Firm-years in non-US panel: {len(panel_nonus)} of {len(df_nonus)} original non-US firm-years")
print(f"({len(df_nonus) - len(panel_nonus)} excluded -- see Step 12.1\'s exclusion list above for why)")

est_mismatch = panel_nonus[panel_nonus["Est_Calendar_Mismatch_N"] > 0]
print(f"\nFirm-years with >=1 estimation-window calendar-mismatch day "
      f"(firm trades, URTH doesn\'t, or vice versa): {len(est_mismatch)} of {len(panel_nonus)}")
if len(est_mismatch):
    display(est_mismatch[["Company", "Fiscal Year", "Estimation N", "Est_Calendar_Mismatch_N"]]
            .sort_values("Est_Calendar_Mismatch_N", ascending=False))

primary_mismatch = panel_nonus[panel_nonus["CAR[-1,+1]_Calendar_Gaps"] > 0]
print(f"\nFirm-years with >=1 calendar-mismatch day INSIDE the primary CAR[-1,+1] window "
      f"(only 3 trading days wide -- a single dropped day is a meaningfully larger share of this "
      f"window than of the ~220-day estimation window): {len(primary_mismatch)} of {len(panel_nonus)}")
if len(primary_mismatch):
    display(primary_mismatch[["Company", "Fiscal Year", "CAR[-1,+1]", "CAR[-1,+1]_Calendar_Gaps"]])
else:
    print("None -- every non-US firm-year\'s primary event window had a matching URTH trading day "
          "for all 3 days.")


## Step 13 — Combined 22-company panel

In [ ]:
# ============================================================
# Step 13.1 -- Combine the US (Part 1) and non-US (Part 3) panels into
# one 22-company panel. Non-US-only diagnostic columns
# (Est_Calendar_Mismatch_N, CAR[-1,+1]_Calendar_Gaps) are kept, filled
# with 0 for the US rows (genuinely near-zero mismatch expected there,
# not imputed as unknown).
# ============================================================
panel_us_tagged = panel[display_cols].copy()
panel_us_tagged["Region"] = "United States"
panel_us_tagged["Est_Calendar_Mismatch_N"] = 0
panel_us_tagged["CAR[-1,+1]_Calendar_Gaps"] = 0

region_by_canonical = (
    df_nonus.drop_duplicates("Company_canonical")
    .set_index("Company_canonical")["Region"]
    .to_dict()
)
panel_nonus_tagged = panel_nonus.copy()
panel_nonus_tagged["Region"] = panel_nonus_tagged["Company"].map(region_by_canonical)

combined_cols = ["Company", "Region", "Fiscal Year", "Event Date (adjusted)", "Estimation N",
                  "Est_Calendar_Mismatch_N", "Alpha", "Beta", "CAR[-1,+1]", "CAR[-1,+1]_Calendar_Gaps",
                  "CAR[-2,+2]", "CAR[0,+1]", "CAR[0,+5]", "CAR[0,+20]", "|CAR|[-1,+1]"]
panel_full = pd.concat([panel_us_tagged, panel_nonus_tagged], ignore_index=True, sort=False)
panel_full = panel_full[combined_cols].sort_values(["Region", "Company", "Fiscal Year"]).reset_index(drop=True)

n_companies_full = panel_full["Company"].nunique()
print(f"Combined panel: {len(panel_full)} firm-years ({n_companies_full} companies)")
print(panel_full.groupby("Region").size().to_string())

output_path_nonus = "/content/drive/MyDrive/phd/Data/ceo_letter_event_study_panel_nonUS.csv"
panel_nonus_tagged[combined_cols[:1] + combined_cols[2:]].to_csv(output_path_nonus, index=False)
print(f"\nSaved non-US-only panel to: {output_path_nonus}")

output_path_full = "/content/drive/MyDrive/phd/Data/ceo_letter_event_study_panel_full22.csv"
panel_full.to_csv(output_path_full, index=False)
print(f"Saved combined 22-company panel to: {output_path_full}")

display(panel_full)
